In [7]:
from sklearn.metrics import accuracy_score, roc_curve, confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, train_test_split
from tabpfn import TabPFNClassifier
from sklearn.preprocessing import StandardScaler
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
from natsort import natsorted
from tqdm import tqdm
import gc
import glob
from sklearn.decomposition import PCA
gc.collect()
torch.cuda.empty_cache()
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"

In [8]:
def metrics(tn, fp, fn, tp):
    """
    Calculate confusion matrix and various performance metrics.
    Args:
        tn (int): True Negatives
        fp (int): False Positives
        fn (int): False Negatives
        tp (int): True Positives
    Returns:
        cm_df (pd.DataFrame): Confusion matrix as a DataFrame.
        metric_df (pd.DataFrame): Performance metrics as a DataFrame.
    """
    tn = tn.astype(np.float32)
    fp = fp.astype(np.float32)
    fn = fn.astype(np.float32)
    tp = tp.astype(np.float32)

    cm_dict = {'predict\\actual':['Positive', 'Negative']
               ,'Positive':[tp, fn]
               ,'Negative':[fp, tn]}

    cm_df = pd.DataFrame(cm_dict)
    
    accuracy = round((tp + tn) / (tp + tn + fp + fn), 4) if (tp + tn + fp + fn) > 0 else 0  
    sensitivity = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0
    specificity = round(tn / (tn + fp), 4) if (tn + fp) > 0 else 0
    ppv = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0
    npv = round(tn / (tn + fn), 4) if (tn + fn) > 0 else 0
    mcc = round((tp * tn - fp * fn) / np.sqrt(float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn)), 4) if (tp + fp) > 0 and (tp + fn) > 0 and (tn + fp) > 0 and (tn + fn) > 0 else 0

    metric_dict = {'metric':['Accuruacy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'MCC'],
                   'Value':[accuracy, sensitivity, specificity, ppv, npv, mcc]}

    metric_df = pd.DataFrame(metric_dict)

    return cm_df, metric_df

In [9]:
non_mace_dir = r"D:\M143020071\raw_data_result\iSKNA_signal\ch1\sr10000_500_1000_MI_1000pts_win60s_step1s\non_mace"
mace_dir = r"D:\M143020071\raw_data_result\iSKNA_signal\ch1\sr10000_500_1000_MI_1000pts_win60s_step1s\mace"

save_data_dir = r"D:\M143020071\raw_data_result\TabPFN_result\Without_PCA"
save_combine_data_dir = os.path.join(save_data_dir, "combine")

In [10]:
for save_dir in [save_data_dir, save_combine_data_dir]:
    os.makedirs(save_dir, exist_ok=True)

In [11]:
all_data_dict = {}
non_mace_ids, mace_ids = [], []

for file_path in glob.glob(os.path.join(non_mace_dir, "*.npy")):
    file_name = os.path.basename(file_path).replace(".npy", "")
    all_data_dict[file_name] = np.load(file_path)
    non_mace_ids.append(file_name)

for file_path in glob.glob(os.path.join(mace_dir, "*.npy")):
    file_name = os.path.basename(file_path).replace(".npy", "")
    all_data_dict[file_name] = np.load(file_path)
    mace_ids.append(file_name)

GROUP_IDS = np.array(non_mace_ids + mace_ids)
GROUP_LABELS = np.array([0] * len(non_mace_ids) + [1] * len(mace_ids))
print(f"讀取完成！共有 {len(non_mace_ids)} 筆健康資料，{len(mace_ids)} 筆 MI 資料。")

讀取完成！共有 200 筆健康資料，200 筆 MI 資料。


In [12]:
gc.collect()
torch.cuda.empty_cache()

# ================= 4. 模型訓練與預測 =================
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
total_train_tp = total_train_fn = total_train_fp = total_train_tn = 0
total_test_tp = total_test_fn = total_test_fp = total_test_tn = 0

LIMIT_TRAIN_NON_MACE = 10
LIMIT_TRAIN_MACE = 40

for fold, (train_idx, test_idx) in enumerate(skf.split(GROUP_IDS, GROUP_LABELS)):
    print(f"\n[With PCA] ========== 正在訓練 Fold {fold + 1}/5 ==========")
    train_list, test_list, train_ids, test_ids = [], [], [], []

    for train_id in GROUP_IDS[train_idx]:
        data = all_data_dict[train_id][:LIMIT_TRAIN_NON_MACE, :] if train_id in non_mace_ids else all_data_dict[train_id][:LIMIT_TRAIN_MACE, :]
        train_list.append(data.copy())
        train_ids.extend([str(train_id)] * data.shape[0])

    for test_id in GROUP_IDS[test_idx]:
        data = all_data_dict[test_id][:, :]
        test_list.append(data.copy())
        test_ids.extend([str(test_id)] * data.shape[0])
            
    train = np.vstack(train_list)
    test = np.vstack(test_list)
    train_ids_df = pd.DataFrame({'research_id': train_ids})
    test_ids_df = pd.DataFrame({'research_id': test_ids})

    X_train, y_train = train[:, 1:], train[:, 0]
    X_test, y_test = test[:, 1:], test[:, 0]
    
    # 🌟 PCA 降維 (降至 100 維)
    pca = PCA(n_components=100, random_state=42)
    X_train = pca.fit_transform(X_train)
    X_test = pca.transform(X_test)
    
    print(f'Train shape : X={X_train.shape}, y={y_train.shape} | Test shape : X={X_test.shape}, y={y_test.shape}')

    clf = TabPFNClassifier(device='auto', random_state=42)
    clf.fit(X_train, y_train)

    gc.collect(); torch.cuda.empty_cache()

    yhat_train = clf.predict(X_train)
    yhat_test = clf.predict(X_test)

    # 存檔
    y_train_label = pd.concat([train_ids_df.reset_index(drop=True), pd.DataFrame(y_train, columns=['y_train']), pd.DataFrame(yhat_train, columns=['yhat_train'])], axis=1)
    y_test_label = pd.concat([test_ids_df.reset_index(drop=True), pd.DataFrame(y_test, columns=['y_test']), pd.DataFrame(yhat_test, columns=['yhat_test'])], axis=1)
    y_train_label.to_csv(os.path.join(save_data_dir, f'y_train_label_{fold + 1}.csv'), index=False)
    y_test_label.to_csv(os.path.join(save_data_dir, f'y_test_label_{fold + 1}.csv'), index=False)

    # 累計混淆矩陣
    train_tn, train_fp, train_fn, train_tp = confusion_matrix(y_train, yhat_train).ravel()
    total_train_tp += train_tp; total_train_fn += train_fn; total_train_fp += train_fp; total_train_tn += train_tn
    test_tn, test_fp, test_fn, test_tp = confusion_matrix(y_test, yhat_test).ravel()
    total_test_tp += test_tp; total_test_fn += test_fn; total_test_fp += test_fp; total_test_tn += test_tn
    
    gc.collect(); torch.cuda.empty_cache()

# ================= 5. 輸出整體報告 =================
train_cm_df, train_metric_df = metrics(total_train_tn, total_train_fp, total_train_fn, total_train_tp)
test_cm_df, test_metric_df = metrics(total_test_tn, total_test_fp, total_test_fn, total_test_tp)

train_cm_df.to_csv(os.path.join(save_combine_data_dir, "train_cm.csv"), index=False)
train_metric_df.to_csv(os.path.join(save_combine_data_dir, "train_metric.csv"), index=False)
test_cm_df.to_csv(os.path.join(save_combine_data_dir, "test_cm.csv"), index=False)
test_metric_df.to_csv(os.path.join(save_combine_data_dir, "test_metric.csv"), index=False)

print('\n=========== [With PCA] TEST RESULTS ===========')
print(f"confusion matrix of test :\n{test_cm_df.to_string(index=False)}\n")
print(f"metric of test :\n{test_metric_df.to_string(index=False)}\n")



[With PCA] ========== 正在訓練 Fold 1/5 ==========
Train shape : X=(8000, 100), y=(8000,) | Test shape : X=(19280, 100), y=(19280,)

[With PCA] ========== 正在訓練 Fold 2/5 ==========
Train shape : X=(8000, 100), y=(8000,) | Test shape : X=(19280, 100), y=(19280,)

[With PCA] ========== 正在訓練 Fold 3/5 ==========
Train shape : X=(8000, 100), y=(8000,) | Test shape : X=(19280, 100), y=(19280,)

[With PCA] ========== 正在訓練 Fold 4/5 ==========
Train shape : X=(8000, 100), y=(8000,) | Test shape : X=(19280, 100), y=(19280,)

[With PCA] ========== 正在訓練 Fold 5/5 ==========
Train shape : X=(8000, 100), y=(8000,) | Test shape : X=(19280, 100), y=(19280,)

=========== [With PCA] TEST RESULTS ===========
confusion matrix of test :
predict\actual  Positive  Negative
      Positive   39646.0   38744.0
      Negative    8554.0    9456.0

metric of test :
     metric  Value
  Accuruacy 0.5094
Sensitivity 0.8225
Specificity 0.1962
        PPV 0.5058
        NPV 0.5250
        MCC 0.0240

